# Chronic Kidney Disease (CKD) Prediction
## Multimodal Fusion using Longitudinal Labs and Clinical Text

**Submission‑ready Google Colab notebook**

---
**Instructions:**
1. Upload the CSV file when prompted below.
2. Run all cells from top to bottom.
3. Outputs (models, plots, metrics) will be saved automatically.
---

## 📂 Upload Dataset

In [ ]:
from google.colab import files
uploaded = files.upload()

# Get uploaded CSV filename
CSV_PATH = list(uploaded.keys())[0]
print('Using dataset:', CSV_PATH)

## 🧪 Install Dependencies

In [ ]:
!pip install -q xgboost shap joblib scikit-learn pandas numpy matplotlib

## 📦 Imports and Configuration

In [ ]:
import os
import numpy as np
import pandas as pd
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LinearRegression

import xgboost as xgb
import shap
import joblib
import matplotlib.pyplot as plt

OUT_DIR = '/content/ckd_pipeline_output'
RANDOM_STATE = 42
TEST_SIZE = 0.2

os.makedirs(OUT_DIR, exist_ok=True)

## 🛠️ Helper Functions

In [ ]:
def parse_series(series_str):
    pairs = str(series_str).split('|')
    dates, values = [], []
    for p in pairs:
        if ':' not in p:
            continue
        d, v = p.split(':')
        try:
            dates.append(datetime.strptime(d, '%Y-%m-%d'))
            values.append(float(v))
        except:
            continue
    return dates, values

def compute_slope(dates, values):
    if len(values) < 2:
        return 0.0
    X = np.array([(d - dates[0]).days for d in dates]).reshape(-1, 1)
    y = np.array(values)
    return LinearRegression().fit(X, y).coef_[0]

## 📊 Load Dataset

In [ ]:
df = pd.read_csv(CSV_PATH)
print('Rows:', len(df))
df.head()

## 🧬 Feature Engineering

In [ ]:
rows = []
for _, r in df.iterrows():
    e_dates, e_vals = parse_series(r['eGFR_series'])
    c_dates, c_vals = parse_series(r['Creatinine_series'])

    rows.append({
        'eGFR_baseline': e_vals[0] if e_vals else np.nan,
        'eGFR_last': e_vals[-1] if e_vals else np.nan,
        'eGFR_slope_year': compute_slope(e_dates, e_vals) * 365,
        'creatinine_last': c_vals[-1] if c_vals else np.nan,
        'label': int(r['Label_CKD']),
        'notes': str(r['Notes_Summary'])
    })

features = pd.DataFrame(rows)
features.to_csv(f'{OUT_DIR}/engineered_features.csv', index=False)

## 📝 Clinical Text Vectorization

In [ ]:
tfidf = TfidfVectorizer(max_features=200)
X_text = tfidf.fit_transform(features['notes']).toarray()
joblib.dump(tfidf, f'{OUT_DIR}/tfidf_vectorizer.joblib')

## 🤖 Model Training and Evaluation

In [ ]:
X_meta = features.drop(columns=['notes', 'label']).fillna(features.mean())
X = np.hstack([X_meta.values, X_text])
y = features['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)

model = xgb.XGBClassifier(eval_metric='logloss')
model.fit(X_train, y_train)

pred = model.predict(X_test)
proba = model.predict_proba(X_test)[:,1]

print('Accuracy:', accuracy_score(y_test, pred))
print('Precision:', precision_score(y_test, pred))
print('Recall:', recall_score(y_test, pred))
print('F1:', f1_score(y_test, pred))
print('AUC:', roc_auc_score(y_test, proba))

## 🔍 Model Explainability (SHAP)

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test[:200])
shap.summary_plot(shap_values, X_test[:200])

## ✅ Conclusion

This notebook provides a reproducible multimodal CKD prediction pipeline.

**Key strengths:**
- Longitudinal lab trend modeling
- Clinical text fusion
- Interpretable ML using SHAP

The framework is suitable for MS/PhD coursework, thesis work, and clinical ML research.